# DiGToR on FMB - extra seeds (0, 1) for mean +/- std

The seed-42 run already exists (`ckpt_fmb/digtor.pt` + W&B `digtor-ckpt`). This
notebook adds **2 more seeds** so the **3-seed 5A ablation table** can be produced.

This notebook only **trains + pushes checkpoints** (seeds 0, 1). The 5A `mean +/- std`
table itself is computed in `digtor_figures_and_gaps.ipynb`, which pulls all 3
checkpoints (`digtor`, `digtor-seed0`, `digtor-seed1`) from W&B -- so 5A lives in
exactly one place.

Contract that keeps the 3 seeds comparable and seed-42 untouched:
- **Same teachers.** v_only/t_only (seed 42) are reused as fixed KD targets; only
  *digtor* is retrained. -> 2 x ~2h, not 6h.
- **Same test set.** FMB official `test/` is fixed regardless of seed, so all three
  checkpoints are scored on identical held-out data.
- **No clobbering.** Each new seed trains into `ckpt_fmb/seed{N}` and is pushed under
  artifact `digtor-seed{N}-ckpt` (NOT `digtor-ckpt`). The seed-42 checkpoint/artifact
  are read-only here.

Cells below the setup block are new; the 9 setup cells are copied verbatim from
`digtor-fmb.ipynb` so the training recipe matches seed 42 exactly.

In [ ]:
import os
import shutil

repo_name = "DiGToR"

# Nếu thư mục đã tồn tại, xóa đi để chuẩn bị tải mới
if os.path.exists(repo_name):
    print(f"Phát hiện thư mục '{repo_name}' đã tồn tại. Đang xóa...")
    shutil.rmtree(repo_name)
    print("Đã xóa thư mục cũ.")

repo_url = f"https://github.com/nguyenmaiductrong/{repo_name}.git"

print(f"Đang tải repo từ {repo_url}...")
exit_code = os.system(f"git clone {repo_url}")

if exit_code == 0:
    print("Tải thành công. Kiểm tra ở thư mục Output/Working.")
else:
    print("Có lỗi xảy ra khi tải, kiểm tra lại đường dẫn mạng hoặc URL.")

In [ ]:
# repo da duoc clone moi o cell 0 (fresh = latest main); pull nay chi de chac chan
!cd DiGToR && git pull origin main || echo "(pull skipped - fresh clone already latest)"

In [ ]:
cd DiGToR

In [ ]:
# --- Download FMB data from Google Drive and unzip train/test ---
# Data is stored OUTSIDE the cloned repo so a re-clone does not wipe it, and the
# step is idempotent: re-running skips the ~1GB download if the data is ready.
import os, glob, zipfile, shutil, subprocess, sys

DRIVE_URL = "https://drive.google.com/drive/folders/1T_jVi80tjgyHTQDpn-TjfySyW4CK1LlF"
FMB_DIR = "/content/FMB" if os.path.isdir("/content") else os.path.join(
    os.path.dirname(os.getcwd()), "FMB")
_mods = ("Visible", "Infrared", "Label")

def _ready(split):
    return all(os.path.isdir(os.path.join(FMB_DIR, split, m)) for m in _mods)

if _ready("train") and _ready("test"):
    print("FMB already prepared at", FMB_DIR)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown
    dl = os.path.join(os.getcwd(), "_fmb_download")
    shutil.rmtree(dl, ignore_errors=True)
    os.makedirs(dl, exist_ok=True)
    gdown.download_folder(DRIVE_URL, output=dl, quiet=False, use_cookies=False)

    def _find(name):
        hits = glob.glob(os.path.join(dl, "**", name), recursive=True)
        if not hits:
            raise FileNotFoundError(f"{name} not found in the Drive folder")
        return hits[0]

    def _modroot(base):
        # the directory that directly holds Visible/Infrared/Label, whatever the
        # zip's internal nesting is
        for root, _, _ in os.walk(base):
            if all(os.path.isdir(os.path.join(root, m)) for m in _mods):
                return root
        raise FileNotFoundError(f"no Visible/Infrared/Label folder under {base}")

    os.makedirs(FMB_DIR, exist_ok=True)
    for split in ("train", "test"):
        tmp = os.path.join(os.getcwd(), f"_unzip_{split}")
        shutil.rmtree(tmp, ignore_errors=True)
        print(f"unzipping {split}.zip ...")
        with zipfile.ZipFile(_find(f"{split}.zip")) as z:
            z.extractall(tmp)
        dst = os.path.join(FMB_DIR, split)
        shutil.rmtree(dst, ignore_errors=True)
        shutil.move(_modroot(tmp), dst)
        shutil.rmtree(tmp, ignore_errors=True)
    shutil.rmtree(dl, ignore_errors=True)
    print("Prepared FMB at", FMB_DIR)

# The data-detection cell below reads FMB_ROOT first, so this works regardless of
# where the data was staged.
os.environ["FMB_ROOT"] = FMB_DIR
for split in ("train", "test"):
    n = len(glob.glob(os.path.join(FMB_DIR, split, "Label", "*"))) if _ready(split) else 0
    print(f"  {split}: {n} label files")

In [ ]:
# --- Repo + FMB data (auto-detect official train/test split OR legacy flat) ---
# Runs on Colab (Pro / A100) or Kaggle. The previous cell downloads the data and
# sets FMB_ROOT, which is checked first below; the other paths are fallbacks.
import os, sys, glob

REPO = os.getcwd()                 # current dir = the DiGToR repo
assert os.path.isfile(os.path.join(REPO, 'digtor', '__init__.py')),     f'Run this from the DiGToR repo root (no digtor/ package found in {REPO}).'

_mods = ['Visible', 'Infrared', 'Label']

def _is_fmb_root(path):
    flat = all(os.path.isdir(os.path.join(path, s)) for s in _mods)
    split = all(os.path.isdir(os.path.join(path, sp, s)) for sp in ['train', 'test'] for s in _mods)
    return flat or split, split

# Priority: FMB_ROOT (set by the download cell) -> repo/FMB -> Colab /content or
# Drive -> Kaggle inputs.
_candidates = []
if os.environ.get('FMB_ROOT'):
    _candidates.append(os.environ['FMB_ROOT'])
_candidates.append(os.path.join(REPO, 'FMB'))
_candidates += ['/content/FMB', '/content/drive/MyDrive/FMB']
_candidates += glob.glob('/kaggle/input/**/FMB', recursive=True)
_candidates += glob.glob('/kaggle/input/*', recursive=False)

ROOT = None
SPLIT_LAYOUT = False
for _cand in dict.fromkeys(_candidates):
    ok, split = _is_fmb_root(_cand)
    if ok:
        ROOT, SPLIT_LAYOUT = _cand, split
        break
assert ROOT is not None, 'FMB data not found: need flat Visible/Infrared/Label OR official train/test layout.'

print('FMB layout =', 'official train/test (held-out test)' if SPLIT_LAYOUT else 'legacy flat (auto-split)')
print('REPO =', REPO)
print('ROOT =', ROOT)

sys.path.insert(0, REPO)
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# --- Hyper-parameters: Phase-0 hard-seg-loss forced paths ---
# Pinned to the successful Phase-0 UNet rig: a 3-path DiGToR router
# (V-trust / T-rescue / Joint), rel_gate OFF at eval, hard seg_loss on
# force_path='v'/'t' plus teacher KD (--lambda_distill 0.5).
# The digtor.dataset.fmb loader uses the no-leakage data contract:
# FMB/train -> train+val only, FMB/test -> held-out evaluation.
import os
from digtor.models import build_model

# Guardrail: if this fails, the Kaggle clone is not on the expected 3-path code.
_probe = build_model('digtor', base=8)
assert getattr(_probe.router[-1], 'out_channels', None) == 3, 'Expected the 3-path DiGToR router.'
del _probe

H, W, BASE = 384, 512, 32
BS = 8
EPOCHS = 80
DIGTOR_EPOCHS = 80
CKPT, RES = 'ckpt_fmb', 'results_fmb'
AMP = '--amp'                      # set '' to disable mixed precision
LR = '--lr 5e-4'
IGNORE_BG = '--ignore_bg'          # match the G1/reporting convention; set '' to include class 0
CORRUPT = '--corrupt_aug --corrupt_p 0.5'
DISTILL = '--lambda_distill 0.5'   # hard seg-loss forced paths + KD when teachers exist
NOGATE = '--disable_gate'          # train/eval with reliability gate OFF (rel_gate=0)
GAMMA_PRIOR = '--gamma_prior 2.0'
LAMBDA_COST = '--lambda_cost 0.1'
ROUTE_BETA = '--route_beta 0.7'

# DRY RUN: set LIMIT='8' to smoke-test all cells quickly; set '' for full train/test.
LIMIT = ''
LIMIT_ARG = f'--limit {LIMIT}' if LIMIT else ''
if LIMIT:
    os.environ['FMB_LIMIT'] = LIMIT
    EPOCHS = DIGTOR_EPOCHS = 1
else:
    os.environ.pop('FMB_LIMIT', None)

os.makedirs(CKPT, exist_ok=True)
os.makedirs(RES, exist_ok=True)
print('config:', dict(H=H, W=W, BASE=BASE, BS=BS, EPOCHS=EPOCHS,
                      DIGTOR_EPOCHS=DIGTOR_EPOCHS, CKPT=CKPT, RES=RES,
                      LIMIT=LIMIT or 'full', lr=LR,
                      ignore_bg=bool(IGNORE_BG), corrupt_aug=CORRUPT,
                      distill=DISTILL, gate='off', route_beta=ROUTE_BETA))

In [ ]:
# --- Weights & Biases: checkpoint sync (survive Colab disconnects) ---
# Logs metrics and uploads each best checkpoint as a wandb artifact (<mode>-ckpt).
# A dropped session can pull them back (next cell) so finished models are reused
# instead of retrained. Set USE_WANDB=False to disable everything.
import os, subprocess, sys

USE_WANDB = True
WB_PROJECT = 'digtor-fmb'
WB_ENTITY = None            # None = your default wandb entity

if USE_WANDB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wandb'], check=True)
    import wandb
    # Kaggle: add your key as a secret named WANDB_API_KEY. Colab: this prompts
    # once, or set os.environ['WANDB_API_KEY'] = '...' before running.
    try:
        wandb.login()
    except Exception as e:
        print('wandb.login failed -> disabling wandb:', e); USE_WANDB = False

WANDB = (f'--wandb --wandb_project {WB_PROJECT}'
         + (f' --wandb_entity {WB_ENTITY}' if WB_ENTITY else '')) if USE_WANDB else ''
ENTITY_ARG = f'--entity {WB_ENTITY}' if WB_ENTITY else ''
print('wandb:', 'ON' if USE_WANDB else 'OFF', '| WANDB =', repr(WANDB))


In [ ]:
# --- Pull any already-trained checkpoints from wandb into CKPT ---
# Idempotent + failsafe: missing artifacts are skipped. After this, each training
# cell's `[ -f {CKPT}/x.pt ]` guard reuses whatever was recovered instead of
# retraining. Run this first on a fresh runtime to resume a dropped session.
if USE_WANDB:
    !python -m digtor.wandb_ckpt --dataset fmb --pull --project {WB_PROJECT} {ENTITY_ARG} --out {CKPT} --modes v_only t_only fusion digtor
else:
    print('wandb OFF -> skipping checkpoint pull')


In [ ]:
# --- A100 speed knobs (quality-neutral) ---
# channels_last + persistent dataloader workers are always on in the code. Here
# we add torch.compile (fuses the conv graph -> faster GPU step) and match the
# worker count to the host CPU cores so the GPU is never starved waiting on JPEG
# decode. None of this changes the maths, so results are preserved.
import os as _os
WORKERS = f"--workers {min(8, (_os.cpu_count() or 4))}"
COMPILE = '--compile'        # set '' to skip torch.compile (e.g. if it errors)
SPEED = f'{COMPILE} {WORKERS}'.strip()
#
# OPTIONAL, NOT quality-neutral: a bigger batch fills the A100 better but changes
# the optimisation. If you raise BS, scale LR by the same factor (linear scaling
# rule) to keep accuracy, e.g. BS=16 -> LR='--lr 1e-3'. Left at the pinned BS=8.
# BS = 16; LR = '--lr 1e-3'
print('SPEED =', repr(SPEED))


## Extra-seed training (reuses seed-42 teachers, protects seed-42 artifact)

In [ ]:
# --- Extra seeds to train (42 already exists) ---
SEEDS = [0, 1]
print('will produce seeds:', [42] + SEEDS, '| teachers reused from seed 42')
assert os.path.isfile(f'{CKPT}/v_only.pt') and os.path.isfile(f'{CKPT}/t_only.pt'), \
    'teachers missing -- run the wandb-pull setup cell first (needs v_only.pt + t_only.pt)'

In [ ]:
# --- Resume: pull any already-finished per-seed checkpoints so a restarted
# runtime skips them. Artifact name is digtor-seed{N}-ckpt (never digtor-ckpt). ---
if USE_WANDB:
    import wandb
    api = wandb.Api()
    for S in SEEDS:
        dst = f'{CKPT}/seed{S}'
        os.makedirs(dst, exist_ok=True)
        if os.path.isfile(f'{dst}/digtor.pt'):
            print(f'[have] seed {S}: {dst}/digtor.pt'); continue
        ref = f"{(str(WB_ENTITY) + '/') if WB_ENTITY else ''}{WB_PROJECT}/digtor-seed{S}-ckpt:latest"
        try:
            api.artifact(ref, type='model').download(dst)
            print(f'[pulled] {ref} -> {dst}/digtor.pt')
        except Exception as e:
            print(f'[none] seed {S}: no artifact yet ({e})')
else:
    print('wandb OFF -> no resume pull')

In [ ]:
# --- Train digtor for each extra seed, then push under a seed-specific artifact.
# Recipe is byte-for-byte the seed-42 recipe (cell 'Step 5') minus {WANDB}
# (replaced by a manual push) plus --seed. Teachers are loaded, not retrained.
# Push happens right after each seed so a later disconnect can't lose a finished one.
import time
for S in SEEDS:
    OUT = f'{CKPT}/seed{S}'
    os.makedirs(OUT, exist_ok=True)
    p = f'{OUT}/digtor.pt'
    if os.path.isfile(p):
        print(f'[skip-train] seed {S}: {p} exists')
    else:
        cmd = (f'python -m digtor.train --dataset fmb --root {ROOT} --mode digtor '
               f'--epochs {DIGTOR_EPOCHS} --bs {BS} --height {H} --width {W} --base {BASE} '
               f'--out {OUT} --v_ckpt {CKPT}/v_only.pt --t_ckpt {CKPT}/t_only.pt --seed {S} '
               f'{AMP} {LR} {IGNORE_BG} {SPEED} {CORRUPT} {GAMMA_PRIOR} {LAMBDA_COST} '
               f'{ROUTE_BETA} {DISTILL} {NOGATE}')
        print(f'\n>>> seed {S}\n   {cmd}')
        t0 = time.time()
        rc = os.system(cmd)
        print(f'[seed {S}] exit={rc}  ({(time.time()-t0)/60:.1f} min)')
        if rc != 0 or not os.path.isfile(p):
            print(f'[seed {S}] training FAILED -> skip push'); continue
    if USE_WANDB and os.path.isfile(p):
        import wandb
        from digtor.wandb_ckpt import init_run
        run = init_run(WB_PROJECT, name=f'digtor_seed{S}', entity=WB_ENTITY,
                       config={'seed': S, 'role': 'extra-seed', 'teachers': 'seed42'})
        art = wandb.Artifact(f'digtor-seed{S}-ckpt', type='model',
                             metadata={'seed': S})
        art.add_file(p)
        run.log_artifact(art); run.finish()
        print(f'[pushed] digtor-seed{S}-ckpt')